# 03 — Feature Engineering
## Predictive Maintenance Dataset

This notebook builds model-ready features from `predictive_maintenance_clean.csv`.

### Modeling goals
- Binary failure prediction using `Target`
- Failure-type classification using `Failure Type` for failed machines only

`Product ID` is treated as an identifier and is not used as a predictive feature.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Project paths
BASE_DIR = Path("..")

PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

input_path = PROCESSED_DATA_DIR / "predictive_maintenance_clean.csv"
output_path = PROCESSED_DATA_DIR / "predictive_maintenance_features.csv"

print("Input:", input_path)
print("Output:", output_path)

Input: ..\data\processed\predictive_maintenance_clean.csv
Output: ..\data\processed\predictive_maintenance_features.csv


In [3]:
df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (10000, 10)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


## 1. Validate the cleaned dataset

In [4]:
expected_columns = [
    "Product ID",
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Target",
    "Failure Type",
]

missing_expected = [c for c in expected_columns if c not in df.columns]

assert not missing_expected, f"Missing expected columns: {missing_expected}"
assert df["Target"].isin([0, 1]).all()
assert df["Type"].isin(["L", "M", "H"]).all()

print("Validation passed.")
print("Rows:", len(df))
print("Failure rate:", round(df["Target"].mean() * 100, 3), "%")

Validation passed.
Rows: 10000
Failure rate: 3.39 %


## 2. Keep only information that can be used by the model

`Product ID` is an identifier, so it is excluded.

`Target` and `Failure Type` are targets, not input features.

In [5]:
feature_df = df.copy()

feature_df = feature_df.drop(columns=["Product ID"])

feature_df["Failure Type"] = feature_df["Failure Type"].fillna("No Failure")

## 3. Encode machine Type

`Type` is categorical. We use one-hot encoding rather than assigning an artificial numeric order to L/M/H.

In [6]:
feature_df = pd.get_dummies(
    feature_df,
    columns=["Type"],
    prefix="Type",
    dtype=int
)

display(feature_df.head())

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type,Type_H,Type_L,Type_M
0,1,298.1,308.6,1551,42.8,0,0,No Failure,0,0,1
1,2,298.2,308.7,1408,46.3,3,0,No Failure,0,1,0
2,3,298.1,308.5,1498,49.4,5,0,No Failure,0,1,0
3,4,298.2,308.6,1433,39.5,7,0,No Failure,0,1,0
4,5,298.2,308.7,1408,40.0,9,0,No Failure,0,1,0


## 4. Create physically meaningful derived features

These features are derived only from the sensor/operating measurements and do not use `Target` or `Failure Type`.

- Temperature difference: process temperature minus air temperature
- Temperature ratio
- Mechanical power proxy: torque × rotational speed
- Tool-wear intensity relative to speed
- A small set of nonlinear terms for operating stress

In [7]:
feature_df["Temperature Difference [K]"] = (
    feature_df["Process temperature [K]"]
    - feature_df["Air temperature [K]"]
)

feature_df["Temperature Ratio"] = (
    feature_df["Process temperature [K]"]
    / feature_df["Air temperature [K]"]
)

feature_df["Mechanical Power Proxy"] = (
    feature_df["Torque [Nm]"]
    * feature_df["Rotational speed [rpm]"]
)

feature_df["Tool Wear × Torque"] = (
    feature_df["Tool wear [min]"]
    * feature_df["Torque [Nm]"]
)

feature_df["Torque Squared"] = feature_df["Torque [Nm]"] ** 2
feature_df["Rotational Speed Squared"] = (
    feature_df["Rotational speed [rpm]"] ** 2
)

## 5. Check engineered features

In [8]:
engineered_columns = [
    "Temperature Difference [K]",
    "Temperature Ratio",
    "Mechanical Power Proxy",
    "Tool Wear × Torque",
    "Torque Squared",
    "Rotational Speed Squared",
]

display(
    feature_df[engineered_columns].describe().T
)

assert np.isfinite(
    feature_df[engineered_columns].to_numpy()
).all()

print("Engineered features are finite.")

,count,mean,std,min,25%,50%,75%,max
Temperature Difference [K],10000.0,1.000063e+01,1.001094,7.600000e+00,9.300000e+00,9.800000e+00,1.100000e+01,1.210000e+01
Temperature Ratio,10000.0,1.033352e+00,0.003492,1.025116e+00,1.030686e+00,1.032847e+00,1.036777e+00,1.040672e+00
Mechanical Power Proxy,10000.0,5.996715e+04,10193.093881,1.096680e+04,5.310540e+04,5.988390e+04,6.687375e+04,9.998040e+04
Tool Wear × Torque,10000.0,4.314665e+03,2826.567692,0.000000e+00,1.963650e+03,4.012950e+03,6.279000e+03,1.649700e+04
Torque Squared,10000.0,1.698323e+03,808.529407,1.444000e+01,1.102240e+03,1.608010e+03,2.190240e+03,5.867560e+03
Rotational Speed Squared,10000.0,2.399971e+06,620336.536901,1.364224e+06,2.024929e+06,2.259009e+06,2.598544e+06,8.328996e+06


Engineered features are finite.


## 6. Separate the two prediction targets

### Model A — Failure prediction
Predict `Target` (0 = no failure, 1 = failure).

### Model B — Failure type
Predict `Failure Type`, but only among rows where `Target == 1`.

This prevents the failure-type model from learning a meaningless "No Failure" class.

In [9]:
failure_rows = feature_df["Target"] == 1

print("Failure rows:", failure_rows.sum())
print("Failure-type distribution:")
display(
    feature_df.loc[failure_rows, "Failure Type"]
    .value_counts()
)

Failure rows: 339
Failure-type distribution:


Failure Type
Heat Dissipation Failure    112
Power Failure                95
Overstrain Failure           78
Tool Wear Failure            45
No Failure                    9
Name: count, dtype: int64

## 7. Prepare model feature columns

The two target columns remain in the exported feature dataset so that the training notebook can construct each task cleanly.

In [10]:
model_feature_columns = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Type_H",
    "Type_L",
    "Type_M",
    "Temperature Difference [K]",
    "Temperature Ratio",
    "Mechanical Power Proxy",
    "Tool Wear × Torque",
    "Torque Squared",
    "Rotational Speed Squared",
]

assert all(c in feature_df.columns for c in model_feature_columns)

print("Number of model features:", len(model_feature_columns))
print(model_feature_columns)

Number of model features: 14
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_H', 'Type_L', 'Type_M', 'Temperature Difference [K]', 'Temperature Ratio', 'Mechanical Power Proxy', 'Tool Wear × Torque', 'Torque Squared', 'Rotational Speed Squared']


## 8. Final feature dataset

In [11]:
final_columns = model_feature_columns + ["Target", "Failure Type"]

features_df = feature_df[final_columns].copy()

print("Final feature dataset shape:", features_df.shape)
display(features_df.head())

Final feature dataset shape: (10000, 16)


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_H,Type_L,Type_M,Temperature Difference [K],Temperature Ratio,Mechanical Power Proxy,Tool Wear × Torque,Torque Squared,Rotational Speed Squared,Target,Failure Type
0,298.1,308.6,1551,42.8,0,0,0,1,10.5,1.035223,66382.8,0.0,1831.84,2405601,0,No Failure
1,298.2,308.7,1408,46.3,3,0,1,0,10.5,1.035211,65190.4,138.9,2143.69,1982464,0,No Failure
2,298.1,308.5,1498,49.4,5,0,1,0,10.4,1.034888,74001.2,247.0,2440.36,2244004,0,No Failure
3,298.2,308.6,1433,39.5,7,0,1,0,10.4,1.034876,56603.5,276.5,1560.25,2053489,0,No Failure
4,298.2,308.7,1408,40.0,9,0,1,0,10.5,1.035211,56320.0,360.0,1600.00,1982464,0,No Failure


## 9. Save feature-engineered dataset

This file is the handoff from feature engineering to model training.

In [12]:
output_path.parent.mkdir(parents=True, exist_ok=True)

features_df.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print(f"Shape: {features_df.shape}")

Saved: ..\data\processed\predictive_maintenance_features.csv
Shape: (10000, 16)


## 10. Final checks

In [13]:
reloaded = pd.read_csv(output_path)

assert len(reloaded) == len(df)
assert set(model_feature_columns).issubset(reloaded.columns)
assert "Target" in reloaded.columns
assert "Failure Type" in reloaded.columns

print("Feature engineering completed successfully.")
print("Saved dataset:", output_path)

Feature engineering completed successfully.
Saved dataset: ..\data\processed\predictive_maintenance_features.csv
